In [1]:
#import numpy as np
#import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from sklearn import mixture
import scanpy as sc
import pandas as pd
#import pomegranate

In [2]:
#pip install gspa

In [3]:
import numpy as np
import gspa
import scanpy, phate
SEED = 13

def reset_seeds(seed=SEED):
    np.random.seed(seed)
    return seed


2026-03-17 12:07:39.284363: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-17 12:07:40.754165: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [4]:
import graphtools
print('graphtools', graphtools.__version__)
import tensorflow
print('tensorflow', tensorflow.__version__)
import keras
print('keras', keras.__version__)
import numpy as np
print ('numpy', np.__version__)
import sklearn
print ('sklearn', sklearn.__version__)
import scipy
print ('scipy', scipy.__version__)
import tqdm
print ('tqdm', tqdm.__version__)
import scanpy
print ('scanpy', scanpy.__version__)
import phate
print ('phate', phate.__version__)


graphtools 2.1.0
tensorflow 2.13.1
keras 2.13.1
numpy 1.24.3
sklearn 1.3.2
scipy 1.10.1
tqdm 4.67.1
scanpy 1.9.8
phate 1.0.11


In [5]:
###!pip install -e /banach1/wes/LocalMarkers2/GMM_weights/scikit-learn

In [6]:
#pip install ipywidgets scanpy hotspotsc singleCellHaystack https://github.com/ohlerlab/SEMITONES/archive/master.zip igraph

In [7]:
import numpy as np

In [8]:
!nvidia-smi


KeyboardInterrupt



In [ ]:
import os
import sys
sys.path.append('../../../')
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
from tqdm import tqdm

from matplotlib import pyplot as plt
import seaborn as sns
from collections import Counter

In [ ]:
from locat.plotting_and_other_methods import plotgenes, plot_gene_localization_summary, parse_output, train_haystack_logpv, train_hotspot, train_clustering

In [ ]:
#from locat.locat import LOCAT

In [ ]:
import scanpy as sc
data_dir = ""
adata = sc.read_h5ad(data_dir+"method_comp_1.h5ad") 

In [14]:
plt.hist(np.array(adata.X.todense()).flatten()[np.array(adata.X.todense()).flatten()>0], bins=30)

AttributeError: 'numpy.ndarray' object has no attribute 'todense'

In [ ]:
X = adata.X#.todense()

# Summary statistics
print(f"Type: {type(X)}")
print(f"Dtype: {X.dtype}")
print(f"Min: {X.min()}, Max: {X.max()}")
print(f"Mean: {X.mean()}, Median: {np.median(X.A if hasattr(X, 'A') else X)}")

In [17]:
#sc.pp.log1p(adata)

In [18]:
#plt.hist(np.array(adata.X.todense()).flatten()[np.array(adata.X.todense()).flatten()>0], bins=30)

In [19]:
#gnames = pd.read_csv('../../../data/gene_names.txt', header=None)

In [20]:
#adata.var_names = gnames[0]
adata.var.head()

""
Gene_0
Gene_1
Gene_2
Gene_3
Gene_4


In [21]:
#cc_phases = pd.read_csv('../../../data/upper_dermal_E14.5_WT_data_cell_cycle_phases.txt', header=None)
##adata.var["cc"] = cc_phases
##adata.var.head()
##cc_phases
#adata.obs["cc"] = np.array(cc_phases.iloc[:,1])

In [22]:
#c_type = pd.read_csv('../../../data/upper_dermal_E14.5_WT_data_cell_type_labels.txt', header=None)
##adata.var["cc"] = cc_phases
##adata.var.head()
##cc_phases
#adata.obs["ctype"] = np.array(c_type.iloc[:,1])
#adata.obs.head()

In [23]:
#adata.obsm["pca"] = sc.pp.pca(adata.obsm["preprocessed"])

In [24]:
#sc.pp.neighbors(adata, n_pcs=50, use_rep="pca")

In [25]:
adata

AnnData object with n_obs × n_vars = 5000 × 200
    uns: 'neighbors'
    obsm: 'coords'
    obsp: 'connectivities', 'distances'

In [26]:
#adata.uns["genes_used_names"].shape

In [27]:
adata_train = adata.copy()
#adata_train = [:,[i in adata.uns["genes_used_names"] for i in adata.var_names]].copy()

In [28]:
adata_train.X = adata_train.X.toarray()

AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

In [29]:
adata_train

AnnData object with n_obs × n_vars = 5000 × 200
    uns: 'neighbors'
    obsm: 'coords'
    obsp: 'connectivities', 'distances'

## Fit GSPA

In [30]:
%%time
import gspa
import phate
import numpy as np

# Step 1: get PHATE embedding
reset_seeds(SEED)
phate_op = phate.PHATE(n_components=2, random_state=SEED)
embedding = phate_op.fit_transform(adata_train.X)

# Step 2: use GSPA to score each gene as a singleton set
gspa_op = gspa.GSPA()
gspa_op.construct_graph(adata_train.X)
gspa_op.build_diffusion_operator()
gspa_op.build_wavelet_dictionary()

# Embed gene signals from wavelet dictionary
gene_signals = adata_train.X.T # embed all measured genes
gene_ae, gene_pc = gspa_op.get_gene_embeddings(gene_signals)
gene_localization = gspa_op.calculate_localization()

Calculating PHATE...
  Running PHATE on 5000 observations and 200 variables.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 12.19 seconds.
    Calculating KNN search...
    Calculated KNN search in 3.49 seconds.
    Calculating affinities...


/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:810: RuntimeWarning: Detected zero distance between 4141 pairs of samples. Consider removing duplicates to avoid errors in downstream processing.
  warnings.warn(
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:506: RuntimeWarning: overflow encountered in power
  weights = np.exp(-np.power(scaled, decay))
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:505: RuntimeWarning: divide by zero encountered in divide
  scaled = distances_i / bw
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:505: RuntimeWarning: invalid value encountered in divide
  scaled = distances_i / bw
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:540: RuntimeWarning: invalid value encountered in divide
  scaled = distances_i / bw


    Calculated affinities in 11.17 seconds.
  Calculated graph and diffusion operator in 27.17 seconds.
  Calculating landmark operator...
    Calculating SVD...


/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/base.py:554: RuntimeWarning: K should have a non-zero diagonal
  warnings.warn("K should have a non-zero diagonal", RuntimeWarning)


    Calculated SVD in 43.92 seconds.
    Calculating KMeans...
    Calculated KMeans in 30.32 seconds.
  Calculated landmark operator in 74.28 seconds.
  Calculating optimal t...
    Automatically selected t = 4
  Calculated optimal t in 8.21 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 0.15 seconds.
  Calculating metric MDS...
  Calculated metric MDS in 0.57 seconds.
Calculated PHATE in 110.83 seconds.
Calculating PCA...
Calculated PCA in 5.57 seconds.
Calculating KNN search...
Calculated KNN search in 1.48 seconds.
Calculating affinities...


...

Calculated affinities in 5.58 seconds.


/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/base.py:554: RuntimeWarning: K should have a non-zero diagonal
  warnings.warn("K should have a non-zero diagonal", RuntimeWarning)
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/base.py:554: RuntimeWarning: K should have a non-zero diagonal
  warnings.warn("K should have a non-zero diagonal", RuntimeWarning)
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/base.py:554: RuntimeWarning: K should have a non-zero diagonal
  warnings.warn("K should have a non-zero diagonal", RuntimeWarning)
100%|██████████████████████████████████████████████████████████████████| 6/6 [09:14<00:00, 92.38s/it]
2026-02-03 21:51:43.590459: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46404 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:e1:00.0, compute capability: 8.6


Epoch 1/100
4/6 [===================>..........] - ETA: 0s - loss: 0.2199

2026-02-03 21:51:46.320190: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


6/6 [==============================] - 2s 82ms/step - loss: 0.2193 - val_loss: 0.4570
Epoch 2/100
6/6 [==============================] - 1s 119ms/step - loss: 0.1919 - val_loss: 0.4440
Epoch 3/100
6/6 [==============================] - 0s 70ms/step - loss: 0.1795 - val_loss: 0.4397
Epoch 4/100
6/6 [==============================] - 0s 77ms/step - loss: 0.1698 - val_loss: 0.4385
Epoch 5/100
6/6 [==============================] - 1s 105ms/step - loss: 0.1606 - val_loss: 0.4381
Epoch 6/100
6/6 [==============================] - 0s 78ms/step - loss: 0.1512 - val_loss: 0.4397
Epoch 7/100
6/6 [==============================] - 1s 103ms/step - loss: 0.1413 - val_loss: 0.4401
Epoch 8/100
6/6 [==============================] - 1s 106ms/step - loss: 0.1314 - val_loss: 0.4400
Epoch 9/100
6/6 [==============================] - 1s 93ms/step - loss: 0.1215 - val_loss: 0.4404
Epoch 10/100
6/6 [==============================] - 0s 82ms/step - loss: 0.1113 - val_loss: 0.4408
Epoch 11/100
6/6 [=========

In [31]:
np.array(adata_train.var_names)
np.array(adata_train.var_names)[np.argsort(gene_localization)]

array(['Gene_64', 'Gene_94', 'Gene_31', 'Gene_46', 'Gene_74', 'Gene_9',
       'Gene_42', 'Gene_52', 'Gene_68', 'Gene_25', 'Gene_40', 'Gene_47',
       'Gene_91', 'Gene_36', 'Gene_48', 'Gene_21', 'Gene_56', 'Gene_65',
       'Gene_8', 'Gene_88', 'Gene_5', 'Gene_1', 'Gene_35', 'Gene_43',
       'Gene_73', 'Gene_6', 'Gene_32', 'Gene_82', 'Gene_50', 'Gene_55',
       'Gene_15', 'Gene_62', 'Gene_20', 'Gene_58', 'Gene_45', 'Gene_80',
       'Gene_54', 'Gene_3', 'Gene_85', 'Gene_71', 'Gene_34', 'Gene_39',
       'Gene_2', 'Gene_33', 'Gene_4', 'Gene_16', 'Gene_17', 'Gene_59',
       'Gene_11', 'Gene_27', 'Gene_7', 'Gene_30', 'Gene_37', 'Gene_44',
       'Gene_38', 'Gene_66', 'Gene_22', 'Gene_28', 'Gene_81', 'Gene_60',
       'Gene_13', 'Gene_10', 'Gene_29', 'Gene_61', 'Gene_19', 'Gene_57',
       'Gene_23', 'Gene_12', 'Gene_49', 'Gene_0', 'Gene_18', 'Gene_14',
       'Gene_26', 'Gene_41', 'Gene_24', 'Gene_103', 'Gene_78', 'Gene_53',
       'Gene_98', 'Gene_115', 'Gene_110', 'Gene_79', 'Gene_6

In [32]:
#scanpy.external.pl.phate(gene_adata, color=['GSPA_localization'], cmap='PuBuGn', sort_order=False,
#                      vmax=np.percentile(gene_adata.obs['GSPA_localization'], 99.5))

In [33]:
np.savez(data_dir+"gspa_output_comp1_repro.npz", var_names=np.array(adata_train.var_names), gene_localization=gene_localization)

In [34]:
data = np.load(data_dir+"gspa_output_comp1_repro.npz", allow_pickle=True)
var_names = data["var_names"]
gene_localization = data["gene_localization"]

In [35]:
gene_localization[np.argsort(gene_localization)]

array([3.57957314, 3.62417563, 3.62526646, 3.62954878, 3.664934  ,
       3.68158637, 3.68802677, 3.69272615, 3.69356754, 3.69988267,
       3.71165572, 3.71949253, 3.74328306, 3.74407707, 3.74909069,
       3.75015025, 3.75982537, 3.76232257, 3.76493801, 3.77014849,
       3.78369958, 3.79549084, 3.79843123, 3.80000245, 3.80371162,
       3.81410365, 3.81906805, 3.84323589, 3.84517182, 3.85669433,
       3.86192652, 3.87282966, 3.87392183, 3.87643895, 3.87866946,
       3.88207398, 3.88439056, 3.89516387, 3.90887709, 3.91340518,
       3.91694261, 3.93152834, 3.93199048, 3.93332372, 3.94237432,
       3.9512075 , 3.95726773, 3.95727783, 3.96071892, 3.96221264,
       3.96255068, 3.96293431, 3.97921952, 3.98647546, 3.98870257,
       3.99847582, 4.00129097, 4.00631052, 4.02650904, 4.03677509,
       4.03920603, 4.04364469, 4.04753607, 4.0525398 , 4.05327782,
       4.06476548, 4.07823565, 4.09286208, 4.09634036, 4.09878804,
       4.13325322, 4.16845897, 4.1800215 , 4.24216598, 4.26031

In [47]:
np.array(adata_train.var_names)[np.argsort(gene_localization)]

array(['Rplp2', 'Rpl41', 'Tpt1', ..., 'Ccn3', 'Mfap5', 'Aspn'],
      dtype=object)

# Next Simulation

In [37]:
import scanpy as sc
data_dir = ""
adata = sc.read_h5ad(data_dir+"method_comp_2.h5ad") 

In [38]:
%%time
import gspa
import phate
import numpy as np

# Step 1: get PHATE embedding
reset_seeds(SEED)
phate_op = phate.PHATE(n_components=2, random_state=SEED)
embedding = phate_op.fit_transform(adata_train.X)

# Step 2: use GSPA to score each gene as a singleton set
gspa_op = gspa.GSPA()
gspa_op.construct_graph(adata_train.X)
gspa_op.build_diffusion_operator()
gspa_op.build_wavelet_dictionary()

# Embed gene signals from wavelet dictionary
gene_signals = adata_train.X.T # embed all measured genes
gene_ae, gene_pc = gspa_op.get_gene_embeddings(gene_signals)
gene_localization = gspa_op.calculate_localization()

Calculating PHATE...
  Running PHATE on 5000 observations and 200 variables.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 7.58 seconds.
    Calculating KNN search...
    Calculated KNN search in 1.66 seconds.
    Calculating affinities...


/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:810: RuntimeWarning: Detected zero distance between 4121 pairs of samples. Consider removing duplicates to avoid errors in downstream processing.
  warnings.warn(
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:506: RuntimeWarning: overflow encountered in power
  weights = np.exp(-np.power(scaled, decay))
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:505: RuntimeWarning: divide by zero encountered in divide
  scaled = distances_i / bw
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:505: RuntimeWarning: invalid value encountered in divide
  scaled = distances_i / bw
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:540: RuntimeWarning: invalid value encountered in divide
  scaled = distances_i / bw


    Calculated affinities in 6.92 seconds.
  Calculated graph and diffusion operator in 16.42 seconds.
  Calculating landmark operator...
    Calculating SVD...


/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/base.py:554: RuntimeWarning: K should have a non-zero diagonal
  warnings.warn("K should have a non-zero diagonal", RuntimeWarning)


    Calculated SVD in 24.19 seconds.
    Calculating KMeans...
    Calculated KMeans in 18.00 seconds.
  Calculated landmark operator in 42.24 seconds.
  Calculating optimal t...
    Automatically selected t = 4
  Calculated optimal t in 17.42 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 0.15 seconds.
  Calculating metric MDS...
  Calculated metric MDS in 0.89 seconds.
Calculated PHATE in 77.68 seconds.
Calculating PCA...
Calculated PCA in 6.39 seconds.
Calculating KNN search...
Calculated KNN search in 1.84 seconds.
Calculating affinities...


/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:810: RuntimeWarning: Detected zero distance between 4121 pairs of samples. Consider removing duplicates to avoid errors in downstream processing.
  warnings.warn(
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:506: RuntimeWarning: overflow encountered in power
  weights = np.exp(-np.power(scaled, decay))
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:505: RuntimeWarning: divide by zero encountered in divide
  scaled = distances_i / bw
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:505: RuntimeWarning: invalid value encountered in divide
  scaled = distances_i / bw
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/graphs.py:540: RuntimeWarning: invalid value encountered in divide
  scaled = distances_i / bw


Calculated affinities in 7.05 seconds.


/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/base.py:554: RuntimeWarning: K should have a non-zero diagonal
  warnings.warn("K should have a non-zero diagonal", RuntimeWarning)
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/base.py:554: RuntimeWarning: K should have a non-zero diagonal
  warnings.warn("K should have a non-zero diagonal", RuntimeWarning)
/home/wes/.conda/envs/gspa-env/lib/python3.8/site-packages/graphtools/base.py:554: RuntimeWarning: K should have a non-zero diagonal
  warnings.warn("K should have a non-zero diagonal", RuntimeWarning)
100%|█████████████████████████████████████████████████████████████████| 6/6 [10:26<00:00, 104.39s/it]


Epoch 1/100
6/6 [==============================] - 1s 111ms/step - loss: 0.2193 - val_loss: 0.4570
Epoch 2/100
6/6 [==============================] - 0s 85ms/step - loss: 0.1919 - val_loss: 0.4440
Epoch 3/100
6/6 [==============================] - 0s 78ms/step - loss: 0.1795 - val_loss: 0.4397
Epoch 4/100
6/6 [==============================] - 0s 69ms/step - loss: 0.1698 - val_loss: 0.4385
Epoch 5/100
6/6 [==============================] - 0s 37ms/step - loss: 0.1606 - val_loss: 0.4381
Epoch 6/100
6/6 [==============================] - 0s 84ms/step - loss: 0.1512 - val_loss: 0.4397
Epoch 7/100
6/6 [==============================] - 0s 49ms/step - loss: 0.1413 - val_loss: 0.4401
Epoch 8/100
6/6 [==============================] - 0s 87ms/step - loss: 0.1314 - val_loss: 0.4400
Epoch 9/100
6/6 [==============================] - 0s 75ms/step - loss: 0.1215 - val_loss: 0.4404
Epoch 10/100
6/6 [==============================] - 0s 78ms/step - loss: 0.1113 - val_loss: 0.4408
Epoch 11/100
6/6 [

In [39]:
np.savez(data_dir+"gspa_output_comp2_repro.npz", var_names=np.array(adata_train.var_names), gene_localization=gene_localization)

In [40]:
data = np.load(data_dir+"gspa_output_comp2_repro.npz", allow_pickle=True)
var_names = data["var_names"]
gene_localization = data["gene_localization"]

In [41]:
gene_localization[np.argsort(gene_localization)]

array([3.57957314, 3.62417563, 3.62526646, 3.62954878, 3.664934  ,
       3.68158637, 3.68802677, 3.69272615, 3.69356754, 3.69988267,
       3.71165572, 3.71949253, 3.74328306, 3.74407707, 3.74909069,
       3.75015025, 3.75982537, 3.76232257, 3.76493801, 3.77014849,
       3.78369958, 3.79549084, 3.79843123, 3.80000245, 3.80371162,
       3.81410365, 3.81906805, 3.84323589, 3.84517182, 3.85669433,
       3.86192652, 3.87282966, 3.87392183, 3.87643895, 3.87866946,
       3.88207398, 3.88439056, 3.89516387, 3.90887709, 3.91340518,
       3.91694261, 3.93152834, 3.93199048, 3.93332372, 3.94237432,
       3.9512075 , 3.95726773, 3.95727783, 3.96071892, 3.96221264,
       3.96255068, 3.96293431, 3.97921952, 3.98647546, 3.98870257,
       3.99847582, 4.00129097, 4.00631052, 4.02650904, 4.03677509,
       4.03920603, 4.04364469, 4.04753607, 4.0525398 , 4.05327782,
       4.06476548, 4.07823565, 4.09286208, 4.09634036, 4.09878804,
       4.13325322, 4.16845897, 4.1800215 , 4.24216598, 4.26031

# Next Simulation

In [9]:
data_dir = ""
adata = sc.read_h5ad(data_dir+"sim_subs.h5ad")

In [10]:
adata

AnnData object with n_obs × n_vars = 5572 × 1100
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'ngene_lth', 'ngene_hth', 'mt_hth', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'log_nCount_RNA', 'log_nFeature_RNA', 'RNA_snn_res.0.3', 'seurat_clusters', 'cluster', 'tmp.ident', 'RNA_snn_res.0.5', 'celltype'
    var: 'is_partition1', 'is_partition2', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'neighbors', 'pca'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities'

In [11]:
adata.X = np.array(adata.X.todense())

In [12]:
%%time
import gspa
import phate
import numpy as np

# Step 1: get PHATE embedding
reset_seeds(SEED)
phate_op = phate.PHATE(n_components=2, random_state=SEED)
embedding = phate_op.fit_transform(adata.X)

# Step 2: use GSPA to score each gene as a singleton set
gspa_op = gspa.GSPA()
gspa_op.construct_graph(adata.X)
gspa_op.build_diffusion_operator()
gspa_op.build_wavelet_dictionary()

# Embed gene signals from wavelet dictionary
gene_signals = adata.X.T # embed all measured genes
gene_ae, gene_pc = gspa_op.get_gene_embeddings(gene_signals)
gene_localization = gspa_op.calculate_localization()

Calculating PHATE...
  Running PHATE on 5572 observations and 1100 variables.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 11.92 seconds.
    Calculating KNN search...
    Calculated KNN search in 1.65 seconds.
    Calculating affinities...
    Calculated affinities in 2.43 seconds.
  Calculated graph and diffusion operator in 16.04 seconds.
  Calculating landmark operator...
    Calculating SVD...
    Calculated SVD in 7.94 seconds.
    Calculating KMeans...
    Calculated KMeans in 26.36 seconds.
  Calculated landmark operator in 34.32 seconds.
  Calculating optimal t...
    Automatically selected t = 9
  Calculated optimal t in 30.30 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 0.69 seconds.
  Calculating metric MDS...
  Calculated metric MDS in 9.95 seconds.
Calculated PHATE in 92.50 seconds.
Calculating PCA...
Calculated PCA in 13.34 seconds.
Calculating KNN search...
Calculated KNN search in 1.79 s

100%|██████████████████████████████████████████████████████████████████| 6/6 [08:06<00:00, 81.06s/it]
2026-03-17 13:48:52.575836: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43988 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6
2026-03-17 13:48:52.578581: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 9294 MB memory:  -> device: 1, name: NVIDIA RTX A6000, pci bus id: 0000:24:00.0, compute capability: 8.6
2026-03-17 13:48:52.586512: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 46692 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6
2026-03-17 13:48:52.592384: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0

Epoch 1/100
17/33 [==============>...............] - ETA: 0s - loss: 0.0044

2026-03-17 13:48:55.503277: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


33/33 [==============================] - 2s 15ms/step - loss: 0.0041 - val_loss: 0.0094
Epoch 2/100
33/33 [==============================] - 0s 15ms/step - loss: 0.0030 - val_loss: 0.0082
Epoch 3/100
33/33 [==============================] - 1s 18ms/step - loss: 0.0023 - val_loss: 0.0080
Epoch 4/100
33/33 [==============================] - 0s 8ms/step - loss: 0.0020 - val_loss: 0.0076
Epoch 5/100
33/33 [==============================] - 0s 9ms/step - loss: 0.0018 - val_loss: 0.0070
Epoch 6/100
33/33 [==============================] - 0s 8ms/step - loss: 0.0017 - val_loss: 0.0063
Epoch 7/100
33/33 [==============================] - 0s 5ms/step - loss: 0.0017 - val_loss: 0.0057
Epoch 8/100
33/33 [==============================] - 0s 5ms/step - loss: 0.0016 - val_loss: 0.0055
Epoch 9/100
33/33 [==============================] - 0s 15ms/step - loss: 0.0016 - val_loss: 0.0053
Epoch 10/100
33/33 [==============================] - 0s 6ms/step - loss: 0.0016 - val_loss: 0.0052
Epoch 11/100
33/3

In [13]:
np.savez(data_dir+"gspa_output_subs_comp_repro.npz", var_names=np.array(adata.var_names), gene_localization=gene_localization)

In [14]:
data = np.load(data_dir+"gspa_output_subs_comp_repro.npz", allow_pickle=True)
var_names = data["var_names"]
gene_localization = data["gene_localization"]

In [15]:
gene_localization[np.argsort(gene_localization)]

array([ 0.35536486,  0.36631089,  0.42980165, ...,  9.44580801,
        9.77804889, 10.73927936])

# Next Simulation

In [ ]:
data_dir = ""
adata = sc.read_h5ad(data_dir+"sim_depletion.h5ad")

In [ ]:
adata.X = np.array(adata.X.todense())

In [ ]:
%%time
import gspa
import phate
import numpy as np

# Step 1: get PHATE embedding
reset_seeds(SEED)
phate_op = phate.PHATE(n_components=2, random_state=SEED)
embedding = phate_op.fit_transform(adata.X)

# Step 2: use GSPA to score each gene as a singleton set
gspa_op = gspa.GSPA()
gspa_op.construct_graph(adata.X)
gspa_op.build_diffusion_operator()
gspa_op.build_wavelet_dictionary()

# Embed gene signals from wavelet dictionary
gene_signals = adata.X.T # embed all measured genes
gene_ae, gene_pc = gspa_op.get_gene_embeddings(gene_signals)
gene_localization = gspa_op.calculate_localization()

In [ ]:
type(adata.X), adata.X.dtype,# gene_signals.shape

In [ ]:
adata.X.dtype

In [ ]:
np.savez(data_dir+"gspa_output_depl_comp_repro.npz", var_names=np.array(adata.var_names), gene_localization=gene_localization)

In [ ]:
data = np.load(data_dir+"gspa_output_depl_comp_repro.npz", allow_pickle=True)
var_names = data["var_names"]
gene_localization = data["gene_localization"]

In [ ]:
gene_localization[np.argsort(gene_localization)]